# Simulate Henon

Generate your custom Henon's dataset with `n_dim` and `n_steps` of your choice

In [1]:
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

def generateHenon(n_steps=10000, n_dim=10):
    X = np.random.rand(n_steps + 1, n_dim)
    a = 1.4
    b = 0.3
    for i in range(n_steps):
        x_next = np.zeros_like(X[i])
        for j in range(len(x_next)):
            if j != 0:
                x_next[j] = a - (b * X[i][j-1] + (1 - b) * X[i][j])**2 + b * X[i-1][j]
            else:
                x_next[j] = a - X[i][j] ** 2 + b * X[i-1][j]
        X[i+1] = x_next

    parent = os.path.abspath('')
    filename = f"henon_{n_steps}_{n_dim}.npy"
    np.save(os.path.join(parent, 'datasets', filename), X.T)

    return X

X = generateHenon(n_steps=10000, n_dim=10)

# Check if any element has become Infinity or very large
print(np.min(X), np.max(X))

-2.0351359463531384 1.9957450150261005


# Simulate Lorenz-96

Generate your custom Lorenz-96 dataset with `n_dim` and `n_steps` of your choice

In [2]:
from scipy.integrate import odeint
import os, numpy as np

def make_var_stationary(beta, radius=0.97):
    '''Rescale coefficients of VAR model to make stable.'''
    p = beta.shape[0]
    lag = beta.shape[1] // p
    bottom = np.hstack((np.eye(p * (lag - 1)), np.zeros((p * (lag - 1), p))))
    beta_tilde = np.vstack((beta, bottom))
    eigvals = np.linalg.eigvals(beta_tilde)
    max_eig = max(np.abs(eigvals))
    nonstationary = max_eig > radius
    if nonstationary:
        return make_var_stationary(0.95 * beta, radius)
    else:
        return beta

def simulate_var(p, T, lag, sparsity=0.2, beta_value=1.0, sd=0.1, seed=0):
    if seed is not None:
        np.random.seed(seed)

    # Set up coefficients and Granger causality ground truth.
    GC = np.eye(p, dtype=int)
    beta = np.eye(p) * beta_value

    num_nonzero = int(p * sparsity) - 1
    for i in range(p):
        choice = np.random.choice(p - 1, size=num_nonzero, replace=False)
        choice[choice >= i] += 1
        beta[i, choice] = beta_value
        GC[i, choice] = 1

    beta = np.hstack([beta for _ in range(lag)])
    beta = make_var_stationary(beta)

    # Generate data.
    burn_in = 100
    errors = np.random.normal(scale=sd, size=(p, T + burn_in))
    X = np.zeros((p, T + burn_in))
    X[:, :lag] = errors[:, :lag]
    for t in range(lag, T + burn_in):
        X[:, t] = np.dot(beta, X[:, (t-lag):t].flatten(order='F'))
        X[:, t] += + errors[:, t-1]

    return X.T[burn_in:], beta, GC

def lorenz(x, t, F):
    '''Partial derivatives for Lorenz-96 ODE.'''
    p = len(x)
    dxdt = np.zeros(p)
    for i in range(p):
        dxdt[i] = (x[(i+1) % p] - x[(i-2) % p]) * x[(i-1) % p] - x[i] + F
    return dxdt

def generateLorenz96(n_dim, T, F=10.0, delta_t=0.1, sd=0.1, burn_in=1000, seed=42):
    if seed is not None:
        np.random.seed(seed)

    parent = os.path.abspath('')

    # Use scipy to solve ODE.
    x0 = np.random.normal(scale=0.01, size=n_dim)
    t = np.linspace(0, (T + burn_in) * delta_t, T + burn_in)
    X = odeint(lorenz, x0, t, args=(F,))
    X += np.random.normal(scale=sd, size=(T + burn_in, n_dim))

    filename = f"lorenz_{T}_{n_dim}.npy"
    np.save(os.path.join(parent, 'datasets', filename), X.T)
    return X[burn_in:]

X = generateLorenz96(n_dim=10, T=5000)

# Check if any element has become Infinity or very large
print(np.min(X), np.max(X))

-11.079295241708655 16.69624546182404


# Preprocess OBD-II dataset

In [2]:
import numpy as np
import os, copy, pickle, json
import pandas as pd
from sklearn.preprocessing import StandardScaler

parent = os.path.abspath('')
dataset = os.path.join(parent, 'datasets', 'VehicularData(anonymized).csv')

df1 = pd.read_csv(dataset)
print(f"Columns : {df1.columns}")

Columns : Index(['Car_Id', 'Person_Id', 'Trip', 'GPS_Time', 'Device_Time', 'GPS_Long',
       'GPS_Lat', 'GPS_Speed_Ms', 'GPS_HDOP', 'GPS_Bearing', 'Gx', 'Gy', 'Gz',
       'G_Calibrated', 'OBD_KPL_Average', 'OBD_Trip_KPL_Average',
       'OBD_Intake_Air_Temp_C', 'Device_Barometer_M', 'GPS_Altitude_M',
       'OBD_Engine_Load', 'OBD_Fuel_Level', 'GPS_Accuracy_M', 'OBD_Speed_Km',
       'GPS_Speed_Km', 'Device_Trip_Dist_Km', 'OBD_Engine_Coolant_Temp_C',
       'OBD_Engine_RPM', 'OBD_Adapter_Voltage', 'OBD_KPL_Instant',
       'OBD_Fuel_Flow_CCmin', 'Device_Fuel_Remaining',
       'OBD_Ambient_Air_Temp_C', 'OBD_CO2_gkm_Average', 'OBD_CO2_gkm_Instant',
       'Device_Cost_Km_Inst', 'Device_Cost_Km_Trip', 'OBD_Air_Pedal',
       'Context', 'Acceleration_kmhs', 'Reaction_Time', 'Air_Drag_Force',
       'Speed_RPM_Relation', 'KPL_Instant'],
      dtype='object')


C:\Users\ayana\AppData\Local\Temp\ipykernel_25968\2982132043.py:9: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(dataset)


In [3]:
drop_cols = ['GPS_Time', 'Device_Time', 'Trip', 'GPS_Long', 'GPS_Lat', 'GPS_Speed_Ms', 'GPS_HDOP', 'GPS_Bearing', 'Gx', 'Gy', 'Gz', 'G_Calibrated', 'OBD_Trip_KPL_Average',
             'Device_Barometer_M', 'GPS_Altitude_M', 'GPS_Accuracy_M', 'GPS_Speed_Km', 'Device_Trip_Dist_Km', 'OBD_Adapter_Voltage', 'Device_Fuel_Remaining',
             'Device_Cost_Km_Inst', 'Device_Cost_Km_Trip', 'Context', 'Reaction_Time', 'Speed_RPM_Relation', 'KPL_Instant']

for col in drop_cols:
    try:
        df1.drop(col, axis=1, inplace=True)
    except:
        pass
columns = df1.columns
print(f"Columns after dropping :\n{columns}\nCount : {len(columns)}")
print(f"Number of samples : {len(df1)}")

Columns after dropping :
Index(['Car_Id', 'Person_Id', 'OBD_KPL_Average', 'OBD_Intake_Air_Temp_C',
       'OBD_Engine_Load', 'OBD_Fuel_Level', 'OBD_Speed_Km',
       'OBD_Engine_Coolant_Temp_C', 'OBD_Engine_RPM', 'OBD_KPL_Instant',
       'OBD_Fuel_Flow_CCmin', 'OBD_Ambient_Air_Temp_C', 'OBD_CO2_gkm_Average',
       'OBD_CO2_gkm_Instant', 'OBD_Air_Pedal', 'Acceleration_kmhs',
       'Air_Drag_Force'],
      dtype='object')
Count : 17
Number of samples : 91794


In [4]:
data1 = np.array(df1[df1['Car_Id']==1])
data2 = np.array(df1[df1['Car_Id']==2])
num_samples1 = len(data1)
num_samples2 = len(data2)

scalers = {}
for c_idx, col_name in enumerate(columns):
    if col_name in ['Car_Id', 'Person_Id']:
        continue
    # Fit the scaler with CarID=1 instances
    # scaler = MinMaxScaler(feature_range=(0, 1)).fit(data1[:, c_idx].reshape(-1, 1))
    scaler = StandardScaler().fit(data1[:, c_idx].reshape(-1, 1))
    data1[:, c_idx] = scaler.transform(data1[:, c_idx].reshape(-1, 1)).reshape((num_samples1,))

    # Scale CarID=2 instances with trained scalers
    data2[:, c_idx] = scaler.transform(data2[:, c_idx].reshape(-1, 1)).reshape((num_samples2,))

    # Keep scaler instances for future use
    scalers[col_name] = copy.deepcopy(scaler)

# Save to JSON
new_dataset = {}
new_dataset['columns'] = list(columns)
new_dataset['car1'] = data1.tolist()
new_dataset['car2'] = data2.tolist()

with open(os.path.join(parent, 'datasets', 'vehicular_modified.json'), 'w') as fp:
    json.dump(new_dataset, fp)

with open(os.path.join(parent, 'datasets', 'obd2_features_scalers.pk'), 'wb') as fp:
    pickle.dump(scalers, fp, protocol=pickle.HIGHEST_PROTOCOL)

In [5]:
with open(os.path.join(parent, 'datasets', 'vehicular_modified.json'), 'r') as fp:
    df = json.load(fp)
columns = list(df['columns'])
df['car1'] = np.array(df['car1'], dtype=np.float32)
df['car2'] = np.array(df['car2'], dtype=np.float32)
print(f"Columns : {columns}")

data = df['car1']
num_samples, dim = data.shape
print(num_samples, dim)

Columns : ['Car_Id', 'Person_Id', 'OBD_KPL_Average', 'OBD_Intake_Air_Temp_C', 'OBD_Engine_Load', 'OBD_Fuel_Level', 'OBD_Speed_Km', 'OBD_Engine_Coolant_Temp_C', 'OBD_Engine_RPM', 'OBD_KPL_Instant', 'OBD_Fuel_Flow_CCmin', 'OBD_Ambient_Air_Temp_C', 'OBD_CO2_gkm_Average', 'OBD_CO2_gkm_Instant', 'OBD_Air_Pedal', 'Acceleration_kmhs', 'Air_Drag_Force']
85095 17
